# Gestructureerde data: inladen dataset

## Enkel datatype

Er zijn heel veel verschillende bronnen om datasets te vinden. Je kan ze bijvoorbeeld op keras, overheidssites en dergelijke vinden.
Ook zijn er heel wat datasets die standaard geworden zijn voor voorbeelden/modellen te testen. 
Deze datasets worden mee aangeleverd met de verschillende modelling-frameworks en kunnen zo eenvoudig gebruikt worden zonder een extra download uit te voeren.
In deze notebook gaan we werken met deze standaard datasets.

In het eerste voorbeeld gaan we werken met een dataset waarbij alle data reeds numeriek is, meer bepaald gaan we werken met de iris-dataset dat informatie over de bloemen van een aantal iris-soorten bevat.

In [2]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
X_train.shape

(120, 4)

### Pytorch

Om een dataset in lezen en klaar te maken in pytorch moet je werken met een Dataset. Dit kan je doen als volgt:

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

class IrisDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        # nog geen schaling aanwezig hier

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # geef me rij met index idx
        return self.X[idx], self.y[idx]

dataset = IrisDataset(X_train, y_train)
print(len(dataset))

dataloader = DataLoader(dataset, batch_size=10)
print(len(dataloader))

for inputs, labels in dataloader:
    print(inputs.shape, labels.shape)
    #model(inputs).shape
    break

120
12
torch.Size([10, 4]) torch.Size([10])


### Keras

Keras is vooral bedoeld als uitvoerend framework en niet om data in te laden.

## Mixed types

Bovenstaande voorbeelden zijn eenvoudige voorbeelden omdat de data in principe reeds gepreprocessed is of uit slechts 1 datatype bestaat. 
Dit heeft als gevolg dat alle inputs op dezelfde manier verwerkt kunnen worden en er dus geen onderscheid gemaakt moet worden tussen verschillende kolommen.
Data met hetzelfde type kan dus eenvoudig aan een sequentieel neuraal netwerk model gepresenteerd worden.

Indien de dataset echter meerdere types bevatten kan de data niet rechtstreeks aan het neuraal netwerk doorgegeven worden aangezien dit type model enkel met numerieke data kan werken. 
Een standaard dataset waarin dit gebeurd is de titanic dataset.
Hoe dit gebeurd in de verschillende frameworks zie je hieronder

### Pytorch

Aangezien we data binnenkrijgen in een dataset als numpy array/dataframe kunnen we in de constructor of de get_item functie de nodige preprocessing stappen uitvoeren.
Hieronder doe ik het rechtstreeks met pandas, je kan het ook met een sci-kit learn pipeline doen.

**Let op:** Hieronder splits ik niet op train en testdata dus kan ik de preprocessingen stappen in de constructor doen van de dataset.
Indien er echter eerst een splitsing gebeurt in train- en testdata, dan moet je de preprocessing stappen in de test-data doen met de parameters geleerd uit de dataset.
Dit is bijvoorbeeld belangrijk als er een klasse niet aanwezig is in de dataset. Dan zou deze niet gemapt worden en komt de ordinal encoding niet overeen tussen train- en testdata.

In [12]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split

# Load Titanic dataset
titanic = pd.read_csv("https://storage.googleapis.com/tf-datasets/titanic/train.csv")

display(titanic)

class TitanicDataset(Dataset):

    def __init__(self, dataframe):
        # al mijn preprocessing stappen plaats ik hier
        dataframe.fillna({'age': dataframe['age'].median()}, inplace=True)
        dataframe.fillna({'embark_town': dataframe['embark_town'].mode()[0]}, inplace=True) # meest frequente
        # ordinal encoding
        dataframe['sex'] = dataframe['sex'].astype('category').cat.codes
        dataframe['class'] = dataframe['class'].astype('category').cat.codes
        dataframe['embark_town'] = dataframe['embark_town'].astype('category').cat.codes
        
        # omzetten naar tensor
        X=dataframe[['class', 'sex', 'age', 'n_siblings_spouses', 'parch', 'fare']]
        y=dataframe['survived'] # label kolom selecteren

        self.X = torch.tensor(X.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

TitanicDataset(titanic)[1]

,survived,sex,age,n_siblings_spouses,parch,fare,class,deck,embark_town,alone
0,0,male,22.0,1,0,7.2500,Third,unknown,Southampton,n
1,1,female,38.0,1,0,71.2833,First,C,Cherbourg,n
2,1,female,26.0,0,0,7.9250,Third,unknown,Southampton,y
3,1,female,35.0,1,0,53.1000,First,C,Southampton,n
4,0,male,28.0,0,0,8.4583,Third,unknown,Queenstown,y
...,...,...,...,...,...,...,...,...,...,...
622,0,male,28.0,0,0,10.5000,Second,unknown,Southampton,y
623,0,male,25.0,0,0,7.0500,Third,unknown,Southampton,y
624,1,female,19.0,0,0,30.0000,First,B,Southampton,y
625,0,female,28.0,1,2,23.4500,Third,unknown,Southampton,n


(tensor([ 0.0000,  0.0000, 38.0000,  1.0000,  0.0000, 71.2833]), tensor(1))